# Pipeline executing Isolation Foerest Outliers and k-NN Imputing of the missing values

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline

## 1. Importing Data

In [2]:
url2 = 'https://raw.githubusercontent.com/JeroenGuillierme/Project-MDA/main/Data/'

rta_df = pd.read_csv(
    f'{url2}total_df_with_distances.csv')

interventions_data = rta_df[rta_df['Intervention'] == 1]
print('Missing values:\n', interventions_data.isna().sum())

## 2. Importing custom transformers for outlier removal and missing values imputation

**DataFrameSplitter**

* Splits dataframe in two, one containing NaNs and one without NaNs

**IsolationForestFilter**

* The custom transformer IsolationForestOutlierRemoval allows the Isolation Forest to be integrated into a scikit-learn pipeline by providing both fit and transform methods.
* It removes the outliers detected by the IsolationForest and passes the cleaned data to the next step (like KNN imputation).
* This approach ensures that you can streamline the entire preprocessing workflow, including outlier removal and imputation, in a single pipeline.

**KNNImputerByGroup**

* Splits the data per vector type.
* Imputes missing values based on selected feautures: Latitude, Longitude, distance to specified vector type and T3-T0
* Concatenates groups back together

In [3]:
from CustomTransformer import DataFrameSplitter, IsolationForestFilter, KNNImputerByGroup

## 3. Creating Pipeline for each group

In [4]:
pipeline = Pipeline(steps=[
    ('outlier_removal', IsolationForestOutlierRemoval()), # Step 1: Remove outliers
    ('knn_imputation', ImputationByVectorType()) # Step 2: Impute missing values
])



In [5]:
# Run the Pipeline
rta_ready = pipeline.fit_transform(interventions_data)
print('Missing values after imputation:\n', rta_ready.isna().sum())

In [ ]:
# set seed for allowing multiple runs with same outcome
np.random.seed(42)

# Splitting the data into groups based on 'Vector type'
g1 = interventions_data[interventions_data['Vector type'] == 'Ambulance'].reset_index(drop=True)
g2 = interventions_data[interventions_data['Vector type'] == 'MUG'].reset_index(drop=True)
g3 = interventions_data[interventions_data['Vector type'] == 'PIT'].reset_index(drop=True)
g4 = interventions_data[interventions_data['Vector type'].isna()].reset_index(drop=True)

# Create pipeline for each group
pipeline_ambulance = Pipeline([
    ('outlier_removal', IsolationForestOutlierRemoval()),  # Step 1: Remove outliers
    ('knn_imputation', ImputationByVectorType(n_neighbors=5))          # Step 2: Impute missing values
])

pipeline_mug = Pipeline([
    ('outlier_removal', IsolationForestOutlierRemoval()),
    ('knn_imputation', KNNImputer(n_neighbors=5))
])

pipeline_pit = Pipeline([
    ('outlier_removal', IsolationForestOutlierRemoval()),
    ('knn_imputation', KNNImputer(n_neighbors=5))
])

# Apply the pipelines to each group
g1_imputed = pd.DataFrame(pipeline_ambulance.fit_transform(g1[['Latitude', 'Longitude', 'distance_to_ambulance', 'T3-T0']]), 
                          columns=['Latitude', 'Longitude', 'distance_to_ambulance', 'T3-T0'])

g2_imputed = pd.DataFrame(pipeline_mug.fit_transform(g2[['Latitude', 'Longitude', 'distance_to_mug', 'T3-T0']]), 
                          columns=['Latitude', 'Longitude', 'distance_to_mug', 'T3-T0'])

g3_imputed = pd.DataFrame(pipeline_pit.fit_transform(g3[['Latitude', 'Longitude', 'distance_to_pit', 'T3-T0']]), 
                          columns=['Latitude', 'Longitude', 'distance_to_pit', 'T3-T0'])

In [ ]:
g1.loc[:,'T3-T0'] = g1_imputed['T3-T0']
g2.loc[:,'T3-T0'] = g2_imputed['T3-T0']
g3.loc[:,'T3-T0'] = g3_imputed['T3-T0']

rta_ready = pd.concat([g1, g2, g3], axis=0, ignore_index=True) # Group 4 isn't concatenated, leaving these observations behind for further analysis
print('Missing values:\n', rta_ready.isna().sum())